# 02. Limpieza, normalizacion y dataset para RAG

Notebook para consumir el dataset crudo y producir:
- `data/processed/sicoes_convocatorias_clean.parquet`
- `data/rag/sicoes_convocatorias_rag.parquet`
- exportaciones `csv` solo como apoyo local


In [1]:
import html
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "data"

RAW_PARQUET_PATH = DATA_DIR / "raw" / "sicoes_convocatorias_raw.parquet"
PROCESSED_DIR = DATA_DIR / "processed"
RAG_DIR = DATA_DIR / "rag"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAG_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PARQUET_PATH = PROCESSED_DIR / "sicoes_convocatorias_clean.parquet"
PROCESSED_CSV_EXPORT_PATH = PROCESSED_DIR / "sicoes_convocatorias_clean.csv"
RAG_PARQUET_PATH = RAG_DIR / "sicoes_convocatorias_rag.parquet"
RAG_CSV_EXPORT_PATH = RAG_DIR / "sicoes_convocatorias_rag.csv"

BASE_URL = "https://www.sicoes.gob.bo"
DATE_RE = re.compile(r"^\d{2}/\d{2}/\d{4}$")
CUCE_RE = re.compile(r"^\d{2}-\d{4}-\d{2}-\d{6,}-\d-\d$")


In [2]:
raw_df = pd.read_parquet(RAW_PARQUET_PATH)
print(raw_df.shape)
raw_df.head()


(1433, 15)


,draw,row_in_page,raw_record_json,raw_tS2RctxC,raw_MEofQhJH,raw_XHoqQ6Sh,raw_z6Ha6SrA,raw_JR4izTcA,raw_NauUu4n9,raw_stqNTYZx,raw_rCBg1eTm,raw_HMiKcXCk,raw_yVhdcpUY,raw_nkkNSuMZ,raw_UPt9HmhG
0,1,0,"{""tS2RctxC"": ""26-1101-00-1669685-1-1"", ""MEofQh...",26-1101-00-1669685-1-1,Gobierno Autonomo Municipal De Sucre,Obras,ANPP,Construccion Escalinata J.V. Patria Nueva,Si,26/06/2026,08/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('oUihJUm...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
1,1,1,"{""tS2RctxC"": ""26-0417-03-1669697-1-1"", ""MEofQh...",26-0417-03-1669697-1-1,Caja Nacional De Salud Regional Santa Cruz,Bienes,CNC,Adquisicion De Medicamentos Regional Santa Cr...,Si,26/06/2026,01/07/2026,Vigente,,%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
2,1,2,"{""tS2RctxC"": ""26-0417-03-1669695-1-1"", ""MEofQh...",26-0417-03-1669695-1-1,Caja Nacional De Salud Regional Santa Cruz,Bienes,CNC,Adquisicion De Medicamentos Para Abastecimient...,Si,26/06/2026,01/07/2026,Vigente,,%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
3,1,3,"{""tS2RctxC"": ""26-0418-07-1669627-1-1"", ""MEofQh...",26-0418-07-1669627-1-1,Caja Petrolera De Salud Administracion Departa...,Bienes,ANPP,Adquisicion De Tacrolimus. Unidad De Farmacia ...,Si,26/06/2026,08/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('/VAPIi/...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...
4,1,4,"{""tS2RctxC"": ""26-1205-00-1647943-2-1"", ""MEofQh...",26-1205-00-1647943-2-1,Gobierno Autonomo Municipal De El Alto,Obras,ANPE,Const. Piso De Acera En Area De Equipamiento S...,Si,26/06/2026,07/07/2026,Vigente,"<a href='#' onClick=""descargarArchivo('e6Vbem9...",%3C%61%20%68%72%65%66%3D%27%23%27%20%6F%6E%63%...,<a onclick=irFicha('/portal/contrataciones/fic...


In [3]:
def clean_text(value):
    if value is None:
        return ""
    value = str(value)
    value = html.unescape(value)
    soup = BeautifulSoup(value, "html.parser")
    text = soup.get_text(" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()


def extract_ficha_url(value):
    if not value:
        return ""
    value = html.unescape(str(value))
    match = re.search(r"fichaProceso\.php\?cp=([^'\") ]+)", value)
    if match:
        return f"{BASE_URL}/portal/contrataciones/ficha/fichaProceso.php?cp={match.group(1)}"
    return ""


def extract_file_labels(value):
    if not value:
        return ""
    value = html.unescape(str(value))
    soup = BeautifulSoup(value, "html.parser")
    labels = [a.get_text(" ", strip=True) for a in soup.find_all("a")]
    return "; ".join([item for item in labels if item])


def to_iso_date(value):
    value = clean_text(value)
    if not value:
        return ""
    try:
        return datetime.strptime(value, "%d/%m/%Y").strftime("%Y-%m-%d")
    except ValueError:
        return value


def parse_record(raw):
    values = list(raw.values()) if isinstance(raw, dict) else list(raw)
    cleaned = [clean_text(v) for v in values]

    cuce = next((x for x in cleaned if CUCE_RE.match(x)), "")
    dates = [x for x in cleaned if DATE_RE.match(x)]

    tipos = {"Bienes", "Obras", "Servicios Generales", "Consultoria", "Consultoría"}
    modalidades = {"ANPE", "CNC", "CM", "OF", "LP", "LPN", "LPI", "ANPP", "CD", "EX"}
    si_no = {"Si", "No", "Sí"}

    tipo = next((x for x in cleaned if x in tipos), "")
    modalidad = next((x for x in cleaned if x in modalidades), "")
    subasta = next((x for x in cleaned if x in si_no), "")
    estado = next((x for x in cleaned if x in {"Vigente", "Cerrado", "Desierto", "Cancelado"}), "")

    entidad = cleaned[1] if len(cleaned) > 1 else ""
    objeto = cleaned[2] if len(cleaned) > 2 else ""

    if not objeto or objeto == cuce or objeto in tipos:
        candidatos = [
            x
            for x in cleaned
            if len(x) > 30 and not x.startswith("http") and "descargarArchivo" not in x and "fichaProceso.php" not in x
        ]
        objeto = candidatos[0] if candidatos else objeto

    raw_archivos = next((str(v) for v in values if "descargarArchivo" in str(v)), "")
    raw_ficha = next((str(v) for v in values if "fichaProceso.php" in str(v)), "")
    fecha_publicacion = dates[0] if len(dates) > 0 else ""
    fecha_presentacion = dates[1] if len(dates) > 1 else ""

    return {
        "cuce": cuce,
        "entidad": entidad,
        "tipo_contratacion": "Consultoría" if tipo == "Consultoria" else tipo,
        "modalidad": modalidad,
        "objeto_contratacion": objeto,
        "subasta": "Sí" if subasta == "Si" else subasta,
        "fecha_publicacion": fecha_publicacion,
        "fecha_presentacion": fecha_presentacion,
        "fecha_publicacion_iso": to_iso_date(fecha_publicacion),
        "fecha_presentacion_iso": to_iso_date(fecha_presentacion),
        "estado": estado,
        "archivos_disponibles": extract_file_labels(raw_archivos),
        "ficha_url": extract_ficha_url(raw_ficha),
    }


def build_texto_rag(row):
    parts = [
        f"CUCE: {row.get('cuce', '')}",
        f"Entidad: {row.get('entidad', '')}",
        f"Tipo de contratación: {row.get('tipo_contratacion', '')}",
        f"Modalidad: {row.get('modalidad', '')}",
        f"Objeto de contratación: {row.get('objeto_contratacion', '')}",
        f"Subasta: {row.get('subasta', '')}",
        f"Estado: {row.get('estado', '')}",
        f"Fecha de publicación: {row.get('fecha_publicacion_iso') or row.get('fecha_publicacion', '')}",
        f"Fecha de presentación: {row.get('fecha_presentacion_iso') or row.get('fecha_presentacion', '')}",
        f"Archivos disponibles: {row.get('archivos_disponibles', '')}",
    ]
    return "\n".join([part for part in parts if not part.endswith(": ")])


In [4]:
clean_rows = []

for item in raw_df.to_dict(orient="records"):
    raw_record = json.loads(item["raw_record_json"])
    row = parse_record(raw_record)
    row["source_draw"] = item.get("draw")
    row["source_row_in_page"] = item.get("row_in_page")
    clean_rows.append(row)

clean_df = pd.DataFrame(clean_rows)
clean_df = clean_df.drop_duplicates(subset=["cuce"], keep="first").reset_index(drop=True)
clean_df["document_id"] = clean_df["cuce"]
clean_df["texto_rag"] = clean_df.apply(build_texto_rag, axis=1)
clean_df["longitud_objeto"] = clean_df["objeto_contratacion"].fillna("").str.len()
clean_df["cantidad_palabras_objeto"] = clean_df["objeto_contratacion"].fillna("").str.split().str.len()

clean_df.head()


,cuce,entidad,tipo_contratacion,modalidad,objeto_contratacion,subasta,fecha_publicacion,fecha_presentacion,fecha_publicacion_iso,fecha_presentacion_iso,estado,archivos_disponibles,ficha_url,source_draw,source_row_in_page,document_id,texto_rag,longitud_objeto,cantidad_palabras_objeto
0,26-1101-00-1669685-1-1,Gobierno Autonomo Municipal De Sucre,Obras,ANPP,Gobierno Autonomo Municipal De Sucre,Sí,26/06/2026,08/07/2026,2026-06-26,2026-07-08,Vigente,Especificaciones Tecnicas; Especificaciones Te...,https://www.sicoes.gob.bo/portal/contratacione...,1,0,26-1101-00-1669685-1-1,CUCE: 26-1101-00-1669685-1-1\nEntidad: Gobiern...,36,5
1,26-0417-03-1669697-1-1,Caja Nacional De Salud Regional Santa Cruz,Bienes,CNC,Caja Nacional De Salud Regional Santa Cruz,Sí,26/06/2026,01/07/2026,2026-06-26,2026-07-01,Vigente,,https://www.sicoes.gob.bo/portal/contratacione...,1,1,26-0417-03-1669697-1-1,CUCE: 26-0417-03-1669697-1-1\nEntidad: Caja Na...,42,7
2,26-0417-03-1669695-1-1,Caja Nacional De Salud Regional Santa Cruz,Bienes,CNC,Caja Nacional De Salud Regional Santa Cruz,Sí,26/06/2026,01/07/2026,2026-06-26,2026-07-01,Vigente,,https://www.sicoes.gob.bo/portal/contratacione...,1,2,26-0417-03-1669695-1-1,CUCE: 26-0417-03-1669695-1-1\nEntidad: Caja Na...,42,7
3,26-0418-07-1669627-1-1,Caja Petrolera De Salud Administracion Departa...,Bienes,ANPP,Caja Petrolera De Salud Administracion Departa...,Sí,26/06/2026,08/07/2026,2026-06-26,2026-07-08,Vigente,Documento Base de Contratacion; Convocatoria,https://www.sicoes.gob.bo/portal/contratacione...,1,3,26-0418-07-1669627-1-1,CUCE: 26-0418-07-1669627-1-1\nEntidad: Caja Pe...,63,8
4,26-1205-00-1647943-2-1,Gobierno Autonomo Municipal De El Alto,Obras,ANPE,Gobierno Autonomo Municipal De El Alto,Sí,26/06/2026,07/07/2026,2026-06-26,2026-07-07,Vigente,Documento Base de Contratacion; Convocatoria; ...,https://www.sicoes.gob.bo/portal/contratacione...,1,4,26-1205-00-1647943-2-1,CUCE: 26-1205-00-1647943-2-1\nEntidad: Gobiern...,38,6


In [5]:
processed_columns = [
    "document_id",
    "cuce",
    "entidad",
    "tipo_contratacion",
    "modalidad",
    "objeto_contratacion",
    "subasta",
    "fecha_publicacion",
    "fecha_presentacion",
    "fecha_publicacion_iso",
    "fecha_presentacion_iso",
    "estado",
    "archivos_disponibles",
    "ficha_url",
    "source_draw",
    "source_row_in_page",
    "longitud_objeto",
    "cantidad_palabras_objeto",
]

rag_columns = processed_columns + ["texto_rag"]

processed_df = clean_df[processed_columns].copy()
rag_df = clean_df[rag_columns].copy()

processed_df.to_parquet(PROCESSED_PARQUET_PATH, index=False)
processed_df.to_csv(PROCESSED_CSV_EXPORT_PATH, index=False, encoding="utf-8-sig")
rag_df.to_parquet(RAG_PARQUET_PATH, index=False)
rag_df.to_csv(RAG_CSV_EXPORT_PATH, index=False, encoding="utf-8-sig")

print("Archivos generados:")
print("-", PROCESSED_PARQUET_PATH)
print("-", RAG_PARQUET_PATH)
print("-", PROCESSED_CSV_EXPORT_PATH, "(auxiliar local)")
print("-", RAG_CSV_EXPORT_PATH, "(auxiliar local)")
print("Filas:", len(rag_df))
print("CUCE duplicados:", rag_df["cuce"].duplicated().sum())
print("Sin texto_rag:", (rag_df["texto_rag"].fillna("") == "").sum())


Archivos generados:
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/processed/sicoes_convocatorias_clean.parquet
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/rag/sicoes_convocatorias_rag.parquet
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/processed/sicoes_convocatorias_clean.csv (auxiliar local)
- /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/data/rag/sicoes_convocatorias_rag.csv (auxiliar local)
Filas: 1429
CUCE duplicados: 0
Sin texto_rag: 0
